In [ ]:
#using StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm=ChatGroq(model="Llama-3.1-8B-instant")
prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
final_prompt=prompt.invoke({"topic":"cars"})
parser = StrOutputParser()


chain = prompt | llm | parser

result = chain.invoke(final_prompt)

print(result)

Why did the car's GPS go to therapy? 

Because it was feeling a little "lost" and had a lot of "traffic" to work through.


In [12]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_template(
"""
Answer the question
{format_instructions}

Return the answer in JSON format with:
name
creator

Language: {language}
"""
)

Json_parser = JsonOutputParser()

chain = json_prompt | llm | Json_parser

res = chain.invoke({
    "language": "Java",
    "format_instructions": Json_parser.get_format_instructions()
})

print(res)

{'name': 'API Response', 'creator': 'AI Assistant'}


Using Pydantic Schemas

In [16]:
from pydantic import BaseModel

class Language(BaseModel):
    name:str
    creator:str

In [23]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=Language)
json_prompt = ChatPromptTemplate.from_template(
"""
Answer the question.

{format_instructions}

Language: {language}
"""
)
chain=json_prompt|llm|parser
result = chain.invoke({
    "language": "Java",
    "format_instructions": parser.get_format_instructions()
})

print(result)
print(type(result))

{'name': 'John Doe', 'creator': 'Jane Doe'}
<class 'dict'>


In [22]:
#Using pydantic output parser

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate

class Language(BaseModel):
    name: str
    creator: str

parser=PydanticOutputParser(pydantic_object=Language)

prompt = ChatPromptTemplate.from_template(
"""
Answer the question.

{format_instructions}

Return ONLY the JSON object.
Do not include any explanation or any other text
just the json obj should be there thats it

Language: {language}
"""
)
chain = prompt | llm | parser
result = chain.invoke({
    "language": "Java",
    "format_instructions": parser.get_format_instructions()
})

print(result)


OutputParserException: Invalid json output: ```java
import org.json.JSONObject;

public class Main {
    public static void main(String[] args) {
        JSONObject obj = new JSONObject();
        obj.put("name", "John Doe");
        obj.put("creator", "Jane Doe");
        System.out.println(obj.toString(4));
    }
}
```
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 